# Domain Adaptation

## Loading the Datasets

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define transformation pipeline for MNIST
transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),                    # TODO: resize to 32x32 to match SVHN
    transforms.Grayscale(num_output_channels=3),    # TODO: convert to 3 channels
    transforms.ToTensor(),                          # TODO: transform to tensor
    transforms.Normalize((0.5,), (0.5,))
])

# Define transformation pipeline for SVHN
transform_svhn = transforms.Compose([
    transforms.ToTensor(),                          # TODO: transform to tensor
    transforms.Normalize((0.5,), (0.5,))
])

# Get train and test sets for MNIST
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform_mnist)
mnist_test  = datasets.MNIST(root="./data", train=False, transform=transform_mnist)

# Get train and test sets for SVHN
svhn_train = datasets.SVHN(root="./data", split="train", download=True, transform=transform_svhn)
svhn_test  = datasets.SVHN(root="./data", split="test", download=True, transform=transform_svhn)

# Define data loaders
mnist_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)        # TODO
mnist_test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False)   # TODO

svhn_loader = DataLoader(svhn_train, batch_size=64, shuffle=True)          # TODO
svhn_test_loader = DataLoader(svhn_test, batch_size=64, shuffle=False)     # TODO

## Model Definition

In [ ]:
# Define a simple CNN for feature extraction
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # TODO
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )

    def forward(self, x):
        return self.net(x) # TODO

# Define a final MLP for classification
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            # TODO
            # 32x32 image pulled through two 2x2 max-pools becomes 8x8. 
            # 64 channels * 8 * 8 = 4096
            nn.Linear(64 * 8 * 8, 128), 
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(x) # TODO

# Instantiate models and move them to right device
feature_extractor = FeatureExtractor().to(device)  # TODO
classifier = Classifier().to(device)               # TODO


Training on MNIST...
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done

Before adaptation:
MNIST accuracy: 0.9912
SVHN accuracy: 0.3368

Domain adaptation...
DA Epoch 1 done | Loss: 0.0205
DA Epoch 2 done | Loss: 0.0163
DA Epoch 3 done | Loss: 0.0145
DA Epoch 4 done | Loss: 0.0095
DA Epoch 5 done | Loss: 0.0094

After adaptation:
MNIST accuracy: 0.9932
SVHN accuracy: 0.3460


0.34603564843269824

## Training Loop

In [ ]:
# Define classification loss function and optimizer
criterion = nn.CrossEntropyLoss()  # TODO
optimizer = optim.Adam(list(feature_extractor.parameters()) + list(classifier.parameters()), lr=1e-3)

# Define standard training loop to train the model over mnist
def train_mnist(epochs=5):
    # TODO: set models to train mode
    feature_extractor.train()
    classifier.train()

    for epoch in range(epochs):
        for x, y in mnist_loader:
            # TODO
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            features = feature_extractor(x)
            preds = classifier(features)
            
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} done")

# Define evaluation function
def evaluate(loader, name="dataset"):
    feature_extractor.eval()
    classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = classifier(feature_extractor(x))
            preds = preds.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    print(f"{name} accuracy: {acc:.4f}")
    return acc

## Domain Adaptation Training

The **Maximum Mean Discrepancy (MMD)** is a metric used to measure distance between two probability distributions based on samples.
We define a kernel function that acts as similarity measure:

$$K(x, y) = \exp\left(-\frac{\|x - y\|^2}{2\sigma^2}\right)$$

This kernel maps data into a high dimensional space to find patterns.
If two points are identical, the result is 1. As points get further apart, the result decays toward 0.
$\sigma$ controls the reach of the kernel: small sigma means points must be very close to be considered similar; a large sigma is more forgiving.

This metric is then used to compute the MMD Loss according to the following formula:

$$\text{MMD}^2(P, Q) = E[K(X, X')] + E[K(Y, Y')] - 2E[K(X, Y)]$$

The first term measures how similar is the first dataset to itself.
The second term measures how similar is the second dataset to itself.
The last term measures how similar is the first dataset to the second one.

**Intuition:** if x and y comes from the same distribution, the similarity between them should be roughly the same as the internal similarities, pushing the loss toward 0.

In [ ]:
# Define MMD Loss function

# Implements Radial Basis Function (RBF) or Gaussian Kernel
def gaussian_kernel(x, y, sigma=1.0):
    x = x.unsqueeze(1)
    y = y.unsqueeze(0)
    # TODO
    dist_sq = torch.sum((x - y) ** 2, dim=-1)
    return torch.exp(-dist_sq / (2 * sigma ** 2))

def mmd_loss(x, y):
    Kxx = gaussian_kernel(x, x).mean()  # TODO: compute similarities between x and x
    Kyy = gaussian_kernel(y, y).mean()  # TODO: compute similarities between y and y
    Kxy = gaussian_kernel(x, y).mean()  # TODO: compute similarities between x and y
    return Kxx + Kyy - 2 * Kxy          # TODO


# Define training for Domain Adaptation
def train_domain_adaptation(epochs=5, lambda_mmd=0.5):
    # TODO: set models to train mode
    feature_extractor.train()
    classifier.train()

    svhn_iter = iter(svhn_loader)

    for epoch in range(epochs):
        for mnist_x, mnist_y in mnist_loader:

            try:
                svhn_x, _ = next(svhn_iter)
            except StopIteration:
                svhn_iter = iter(svhn_loader)
                svhn_x, _ = next(svhn_iter)

            mnist_x, mnist_y = mnist_x.to(device), mnist_y.to(device)
            svhn_x = svhn_x.to(device)

            # TODO: clear out old gradients
            optimizer.zero_grad()

            # source (MNIST)
            f_src = feature_extractor(mnist_x)  # TODO: extract features from mnist_x
            preds = classifier(f_src)           # TODO: apply classifier
            cls_loss = criterion(preds, mnist_y) # TODO

            # target (SVHN)
            f_tgt = feature_extractor(svhn_x)   # TODO

            # MMD
            mmd = mmd_loss(f_src, f_tgt)        # TODO

            loss = cls_loss + lambda_mmd * mmd  # TODO

            # TODO (backward pass and step)
            loss.backward()
            optimizer.step()

        print(f"DA Epoch {epoch+1} done | Loss: {loss.item():.4f}")

## Final Pipeline

In [ ]:
# Run entire pipeline
print("\nTraining on MNIST...")
# TODO
train_mnist(epochs=5)

print("\nBefore adaptation:")
# TODO: evaluate over MNIST
evaluate(mnist_test_loader, name="MNIST")
# TODO: evaluate over SVHN
evaluate(svhn_test_loader, name="SVHN")

print("\nDomain adaptation...")
# TODO
train_domain_adaptation(epochs=5, lambda_mmd=0.5)

print("\nAfter adaptation:")
# TODO
evaluate(mnist_test_loader, name="MNIST")
evaluate(svhn_test_loader, name="SVHN")